# 04 · Immutable out-of-sample validation

This notebook is a read-only client of the Phase 5A study registry. Registration and staged evaluation happen through `afterhours-lab-study`; this notebook neither calculates metrics nor writes database rows.

## Hypothesis and preregistration

Validate the example plan before opening any evidence. The canonical digest commits to the hypothesis, universe, versions, split, metric, analysis, sample gates, assumptions, limitations, and failure modes.

In [ ]:
from pathlib import Path

from afterhours_lab.research import fetch_study_detail
from afterhours_lab.research.notebook import research_pool
from afterhours_lab.studies import load_study_spec

spec = load_study_spec(Path('../studies/delay-retention-example.json'))
print(spec.hypothesis)
print('development:', spec.development.from_date, 'to', spec.development.to_date)
print('OOS:', spec.oos.from_date, 'to', spec.oos.to_date)
print('plan digest:', spec.digest)

## Development → frozen rule → sealed OOS

Run `afterhours-lab-study evaluate-development` outside the notebook to append the development snapshot and frozen typed rule. Only then can `evaluate-oos` materialize the held-out cohort, and it applies that stored rule without refitting. The cell below reads an already persisted study through a database pool enforced as read-only.

In [ ]:
pool = await research_pool()
async with pool.acquire() as conn:
    detail = await fetch_study_detail(conn, spec.study_key, spec.version)
await pool.close()

if detail is None:
    print('Plan is not registered. Use the documented CLI dry-run before writing it.')
else:
    print(detail.plan.hypothesis)
    for result in detail.results:
        print(result.stage, result.frozen_rule, result.result_values, result.decision)

## Decision

Interpret only the persisted conservative decision: `reject`, `refine`, or `paper_trade_candidate`. This is observational research with execution assumptions `none`; it is not a signal, fill model, P&L estimate, or trading recommendation.